In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.6 MB/s eta 0:00:00


In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [4]:
def get_centroids(frame):
  result = model.track(frame, persist=True, verbose=False)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  if result[0].boxes.id is None:
    return []
  ids = result[0].boxes.id.cpu().numpy()
  centroid=[]

  for box,c,tid in zip(boxes,conf,ids):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy,tid))

  return centroid

In [5]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 324ms
Prepared 1 package in 44ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



In [6]:
model = YOLO('yolov8n.pt')
players_position = {}
tid_color ={}
prev_ids = set()
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
frame_count = 0
max_frames = 50

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    curr_ids = { tid for cx, cy, tid in centroid }

    for cx, cy, tid in centroid:
        if tid in players_position:
          players_position[tid].append((cx, cy))

        else:
            players_position[tid] = [(cx, cy)]
    disappeared = prev_ids - curr_ids
    new_ids = curr_ids - prev_ids
    prev_ids = curr_ids

print(players_position)

{np.float32(4.0): [(np.float32(403.42545), np.float32(304.4803)), (np.float32(403.37827), np.float32(304.47476)), (np.float32(403.25494), np.float32(304.3175)), (np.float32(403.2353), np.float32(304.37354)), (np.float32(403.00763), np.float32(304.43262)), (np.float32(402.7884), np.float32(304.25476)), (np.float32(402.71506), np.float32(304.18155)), (np.float32(402.59192), np.float32(304.26025)), (np.float32(402.50482), np.float32(304.28833)), (np.float32(402.3012), np.float32(304.2978)), (np.float32(402.25314), np.float32(304.13364)), (np.float32(402.22784), np.float32(304.01935)), (np.float32(402.1094), np.float32(303.8893)), (np.float32(402.10724), np.float32(304.10516)), (np.float32(402.04507), np.float32(304.35193)), (np.float32(401.86823), np.float32(304.39856)), (np.float32(401.4135), np.float32(304.4666)), (np.float32(401.03134), np.float32(304.42087)), (np.float32(400.35736), np.float32(304.27875)), (np.float32(399.69968), np.float32(303.49225)), (np.float32(399.3828), np.float

In [7]:
players_position = {pid: pos for pid, pos in players_position.items() if len(pos) >= 5}

In [8]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{np.float32(4.0): np.float32(48.62578), np.float32(5.0): np.float32(105.0585), np.float32(6.0): np.float32(49.13876), np.float32(7.0): np.float32(32.692703), np.float32(8.0): np.float32(34.77272), np.float32(9.0): np.float32(49.73112), np.float32(10.0): np.float32(59.196594), np.float32(11.0): np.float32(68.620544)}


In [9]:
def get_jersey_crop(frame, box):
    # cast to int since slicing needs whole numbers, not the floats YOLO gives
    x1, y1, x2, y2 = map(int, box)

    height = y2 - y1
    # only take top 35% of box height to isolate jersey, skip shorts/legs
    new_y2 = int(y1 + (0.35 * height))

    # rows (y) first, then columns (x) — standard image slicing order
    return frame[y1:new_y2, x1:x2]

In [10]:

def get_avg_color(crop):
  return crop.mean(axis = (0,1))

In [11]:
avg_colors = []
result = model.track(frame, persist=True, verbose=False)
boxes = result[0].boxes.xyxy.cpu().numpy()
if result[0].boxes.id is None:
    ids = []
else:
    ids = result[0].boxes.id.cpu().numpy()
for box, tid in zip(boxes, ids):
  crop = get_jersey_crop(frame, box)
  avg_color = get_avg_color(crop)
  avg_colors.append((tid, avg_color))
  tid_color[tid] = avg_color

print(len(avg_colors))
print(avg_colors[0])

7
(np.float32(4.0), array([      106.5,      160.12,      141.41]))


In [12]:
colors_only = [c for tid, c in avg_colors]
data = np.array(colors_only, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

print(compactness, labels, centers)

412.85628843307495 [[1]
 [0]
 [1]
 [1]
 [1]
 [0]
 [0]] [[     83.028      129.15      129.87]
 [     102.46      158.43      138.57]]


In [13]:
player_team = {}
for i, (tid, color) in enumerate(avg_colors):
    if tid not in player_team:
        player_team[tid] = labels[i][0]

print(player_team)

{np.float32(4.0): np.int32(1), np.float32(5.0): np.int32(0), np.float32(6.0): np.int32(1), np.float32(7.0): np.int32(1), np.float32(8.0): np.int32(1), np.float32(10.0): np.int32(0), np.float32(11.0): np.int32(0)}


In [14]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team]= 0
  team_distance[team] += dist

print(team_distance)


{1: np.float32(165.22997), 0: np.float32(232.87564)}


In [15]:
import math
def get_player_speeds(position_history, fps):
  speeds = []
  for i in range(1, len(position_history)):
    prev_point = position_history[i-1]
    current_point = position_history[i]
    x1, y1 = prev_point
    x2, y2 = current_point
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    speed = distance * fps
    speeds.append(speed)
  return speeds

fps = cap.get(cv2.CAP_PROP_FPS)

sample_id = list(players_position.keys())[0]
speeds = get_player_speeds(players_position[sample_id], fps)
print(sample_id)
print(speeds)

4.0
[1.187560536777133, 4.996122974924684, 1.484428454992599, 5.8800657846073845, 7.05772356644887, 2.590582230589338, 3.653553345494794, 2.2877647518007604, 5.095823559593444, 4.276156053734229, 2.9263741910369827, 4.397224819359661, 5.396542514649372, 6.3618706227954664, 4.572344454607629, 11.494408833656838, 9.622494843747448, 17.22005016036044, 25.631149763587086, 17.163400118083374, 19.998510536223215, 44.84860623213012, 53.0806777290839, 32.003075140850996, 42.70367441895196, 65.67460864429307, 15.968258525429423, 19.583233319865837, 45.324273059841694, 39.70515887062561, 48.21050275236621, 45.75876126275396, 31.77255862924352, 29.127258119103494, 31.408786291697428, 44.06358020892277, 50.186150453125734, 44.577084761501865, 24.751981924397402, 22.95861234570377, 25.311331545387432, 40.43402901895114, 26.613840806958528, 22.078576692791763, 42.4978724816692, 47.413306805529395, 48.54014656922995, 39.09607969823921, 28.658233328505485]


In [16]:
def count_sprints(speeds, threshold):

  sprint_count = 0
  was_sprinting = False
  for i in speeds:
    is_sprinting = i > threshold
    if is_sprinting and not was_sprinting :
      sprint_count +=1
    was_sprinting = is_sprinting

  return sprint_count

counts = count_sprints(speeds, 30)
print(counts)

5


In [17]:
def build_player_summary(players_position, player_team, player_distances, fps, sprint_threshold):
  player_summary = {}
  for player_id,pos_history in players_position.items():
    # loop over each tracked player to pull together their stats into one summary
    team = player_team.get(player_id, "unknown")
    if not isinstance(team, str):
      team = team.item()
    distance = player_distances[player_id]

    speed = get_player_speeds(pos_history,fps)
    sprint_count = count_sprints(speed,sprint_threshold)

    player_summary[player_id] = {
        "team" : team,
        "distance" : distance,
        "speed" : speed,
        "sprint_count" : sprint_count
    }

  return player_summary


build_player_summary(players_position, player_team, player_distances, fps, 30)


{np.float32(4.0): {'team': 1,
  'distance': np.float32(48.62578),
  'speed': [1.187560536777133,
   4.996122974924684,
   1.484428454992599,
   5.8800657846073845,
   7.05772356644887,
   2.590582230589338,
   3.653553345494794,
   2.2877647518007604,
   5.095823559593444,
   4.276156053734229,
   2.9263741910369827,
   4.397224819359661,
   5.396542514649372,
   6.3618706227954664,
   4.572344454607629,
   11.494408833656838,
   9.622494843747448,
   17.22005016036044,
   25.631149763587086,
   17.163400118083374,
   19.998510536223215,
   44.84860623213012,
   53.0806777290839,
   32.003075140850996,
   42.70367441895196,
   65.67460864429307,
   15.968258525429423,
   19.583233319865837,
   45.324273059841694,
   39.70515887062561,
   48.21050275236621,
   45.75876126275396,
   31.77255862924352,
   29.127258119103494,
   31.408786291697428,
   44.06358020892277,
   50.186150453125734,
   44.577084761501865,
   24.751981924397402,
   22.95861234570377,
   25.311331545387432,
   40.4